In [ ]:
import torch
import torch.nn.functional as F
from importlib import import_module, reload

def safe_import(module_name, names):
    try:
        module = import_module(module_name)
        module = reload(module)
    except Exception as error:
        print(f"SKIP module {module_name}: {error}")
        return

    for name in names:
        if hasattr(module, name):
            globals()[name] = getattr(module, name)
        else:
            print(f"SKIP {module_name}.{name}: not typed yet")

safe_import("data", [
    "make_regression_data",
    "train_test_split",
    "make_region_table",
    "make_region_rules",
    "make_sparse_regression_data",
    "train_test_split_with_regions",
    "make_region_class_rules",
    "make_sparse_classification_data",
])

safe_import("models", [
    "predict_regression",
    "squared_loss",
    "routed_regression_loss",
    "routed_predict_regression",
    "routed_classification_logits",
    "top2_routed_predict_regression",
])

safe_import("router", [
    "route_topk",
    "random_routes",
    "similarity_routes",
])

safe_import("train", [
    "sgd",
    "l2_penalty",
    "rescale_expert_weights",
    "ReplayBuffer",
])

safe_import("metrics", [
    "accuracy",
    "per_region_mse",
    "confusion_matrix",
])

torch.manual_seed(0)

SKIP data.train_test_split_with_regions: not typed yet
SKIP data.make_region_class_rules: not typed yet
SKIP data.make_sparse_classification_data: not typed yet
SKIP models.routed_regression_loss: not typed yet
SKIP models.routed_predict_regression: not typed yet
SKIP models.routed_classification_logits: not typed yet
SKIP models.top2_routed_predict_regression: not typed yet
SKIP router.route_topk: not typed yet
SKIP router.random_routes: not typed yet
SKIP router.similarity_routes: not typed yet
SKIP train.l2_penalty: not typed yet
SKIP train.rescale_expert_weights: not typed yet
SKIP train.ReplayBuffer: not typed yet
SKIP metrics.accuracy: not typed yet
SKIP metrics.per_region_mse: not typed yet
SKIP metrics.confusion_matrix: not typed yet


Training Loop:

In [ ]:
# Generate synthetic regression data using hidden parameters
X, y, true_w, true_b = make_regression_data(200, 6)


# Split X and y into train and test datasets
X_train, y_train, X_test, y_test = train_test_split(X, y)

# Initialize model params (weight and bias) randomly
w = torch.randn(6, requires_grad=True)
b = torch.zeros((), requires_grad=True)

# Run the training process 50 times over the entire training dataset
for epoch in range(50):
    # Make predictions using current parameters
    y_hat = predict_regression(X_train, w, b)

    # Measure how wrong the predictions are
    loss = squared_loss(y_hat, y_train)

    # Compute gradients of loss with respect to w and b
    loss.backward()

    # Update parameters using the gradients
    sgd([w, b], lr=0.05)

"""
X_train
   ↓
predict with current w,b
   ↓
calculate loss
   ↓
backprop calculates gradients
   ↓
SGD updates w,b
   ↓
Start next epoch using the updated w,b

Repeat for 50 epochs
"""

'\nX_train\n   ↓\npredict with current w,b\n   ↓\ncalculate loss\n   ↓\nbackprop calculates gradients\n   ↓\nSGD updates w,b\n   ↓\nStart next epoch using the updated w,b\n\nRepeat for 50 epochs\n'

Evaluation:

In [ ]:
with torch.no_grad(): # Don't track the parameter updates in the computation graph (to do at optimizer.step())
     
    # Performs manual MSE on training vs test/validation loss
    train_loss = squared_loss(predict_regression(X_train, w, b), y_train)
    test_loss = squared_loss(predict_regression(X_test, w, b), y_test)

print("Train loss:", train_loss.item()) # Convert the scalar tensor into a regular Python number with item() for easier printing (and better decimal precision)
print("Test loss:", test_loss.item()) 
print("Trained weight:", w)
print("Trained bias:", b)
print("True weight:",true_w)
print("True bias:",true_b)

Train loss: 0.012925769202411175
Test loss: 0.013824677094817162
Trained weight: tensor([ 1.9570, -2.9703,  1.5105, -0.0131,  0.4891, -0.9645],
       requires_grad=True)
Trained bias: tensor(0.6824, requires_grad=True)
True weight: tensor([ 2.0000, -3.0000,  1.5000,  0.0000,  0.5000, -1.0000])
True bias: tensor(0.7000)


## 7.8 Step-by-Step Breakdown of Phase 1

* `make_regression_data` creates a world with a known linear rule.

* `train_test_split` separates examples used for learning from examples used for evaluation.

* `w` and `b` are the learnable parameters.

* `requires_grad=True` tells PyTorch to track operations involving those tensors.

* `loss.backward()` computes gradients of the scalar loss with respect to w and b.

* `sgd` moves each parameter opposite its gradient.

* `torch.no_grad()` prevents the parameter update itself from being tracked by autograd.

* `p.grad.zero_()` clears gradients so the next epoch starts cleanly.

# 8.6 Checkpoint Prints

In [ ]:
num_regions = 4
num_features = 6
region_table = make_region_table(num_regions, num_features)
true_W, true_b = make_region_rules(num_regions, num_features)
mixture = torch.tensor([0.25, 0.25, 0.25, 0.25]) # Probabilities shape (4, ) since we have 4 regions

X, y, region_ids = make_sparse_regression_data(
    20, region_table, true_W, true_b, mixture # num_examples is determined at 20
)

print(X.shape)
print(y.shape)
print(region_ids.shape)
print(region_ids[:10]) # Randomly placed with fixed probabilties

torch.Size([20, 6])
torch.Size([20])
torch.Size([20])
tensor([2, 1, 2, 2, 3, 0, 2, 3, 0, 0])


# 9.2 Checkpoint Prints

Data generation:

In [ ]:
num_regions = 4
num_features = 6
region_table = make_region_table(num_regions, num_features)
true_W, true_b = make_region_rules(num_regions, num_features)
mixture = torch.tensor([0.25, 0.25, 0.25, 0.25]) # Probabilities shape (4, ) since we have 4 regions

X, y, region_ids = make_sparse_regression_data(
    500, region_table, true_W, true_b, mixture
)
(
    X_train,
    y_train,
    train_region_ids,
    X_test,
    y_test,
    test_region_ids,
) = train_test_split_with_regions(X, y, region_ids)

Train:

In [ ]:
w = torch.randn(num_features, requires_grad=True)
b = torch.zeros((), requires_grad=True)

for epoch in range(200):
    y_hat = predict_regression(X_train, w, b)
    loss = squared_loss(y_hat, y_train)
    loss.backward()
    sgd([w, b], lr=0.01)

Evaluate:

In [ ]:
with torch.no_grad():
    train_loss = squared_loss(predict_regression(X_train, w, b), y_train)
    test_loss = squared_loss(predict_regression(X_test, w, b), y_test)

print("global train loss:", train_loss.item())
print("global test loss:", test_loss.item())